# Evaluation Tests for Parser Output

These tests evaluate the final parsed JSON dictionary from `assessment_processor.py` using parser-output metrics rather than low-level unit tests. Each section prints a score and a `PASS` / `FAIL` result.

## Setup

Load the parser, parse the sample DOCX file, and define small helper functions for evaluating the resulting JSON dictionary.

In [1]:
from collections import Counter
from pathlib import Path
import re
import sys

from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph


def find_preprocessing_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "assessment_processor.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate preprocessing/assessment_processor.py")


preprocessing_dir = find_preprocessing_dir(Path.cwd().resolve())
if str(preprocessing_dir) not in sys.path:
    sys.path.insert(0, str(preprocessing_dir))

from assessment_processor import AssessmentParser, parse_to_dict

source_parser = AssessmentParser()
sample_path = Path("Myrcia neosmithii_draft_status_Apr2022_v2.docx").resolve()
if not sample_path.exists():
    raise FileNotFoundError(f"Sample file not found: {sample_path}")

parsed = parse_to_dict(str(sample_path))
print(f"Parsed: {sample_path.name}")
print(f"Document title: {parsed['title']}")

Parsed: Myrcia neosmithii_draft_status_Apr2022_v2.docx
Document title: Myrcia neosmithii_draft_status_Apr2022_v2


In [2]:
def score_label(score: float) -> str:
    return "PASS" if score == 1.0 else "FAIL"


def print_score(name: str, score: float, details: str = "") -> None:
    print(f"{score_label(score)}: {name}")
    print(f"Score: {score:.2%}")
    if details:
        print(details)


def flatten_section_paths(node: dict):
    paths = []
    for child in node.get("children", []):
        paths.append(tuple(child.get("path", [])))
        paths.extend(flatten_section_paths(child))
    return paths


def flatten_blocks(node: dict):
    blocks = []
    for block in node.get("blocks", []):
        blocks.append((tuple(node.get("path", [])), block))
    for child in node.get("children", []):
        blocks.extend(flatten_blocks(child))
    return blocks


def find_section(node: dict, path: list[str]):
    if node.get("path") == path:
        return node
    for child in node.get("children", []):
        found = find_section(child, path)
        if found is not None:
            return found
    return None


def collect_text_values(node: dict):
    values = []
    for _, block in flatten_blocks(node):
        if "text" in block:
            values.append(block["text"])
        for item in block.get("items", []):
            values.append(item.get("text", ""))
        for row in block.get("rows", []):
            values.extend(row)
    return values


def collect_rich_values(node: dict):
    values = []
    for _, block in flatten_blocks(node):
        if "text_rich" in block:
            values.append(block["text_rich"])
        for item in block.get("items", []):
            if "text_rich" in item:
                values.append(item["text_rich"])
        for row in block.get("rows_rich", []):
            values.extend(row)
    return values


def collect_style_features(node: dict):
    features = set()
    for _, block in flatten_blocks(node):
        if block.get("type") == "style":
            for style_name, snippets in block.get("data", {}).items():
                for snippet in snippets:
                    features.add((style_name, snippet))
    return features


def tokenize_for_bigrams(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())


def bigram_counts(text_values):
    tokens = tokenize_for_bigrams(" ".join(value for value in text_values if value))
    return Counter(zip(tokens, tokens[1:]))


def collect_docx_plain_text(path: Path):
    doc = Document(str(path))
    values = []
    for item in source_parser._iter_block_items_in_order(doc):
        if isinstance(item, Paragraph):
            text = (item.text or "").strip()
            if text:
                values.append(text)
        elif isinstance(item, Table):
            for row in item.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        text = (paragraph.text or "").strip()
                        if text:
                            values.append(text)
    return values


def collect_json_plain_text_for_bigrams(node: dict):
    values = []
    if node.get("path") and node.get("title"):
        values.append(node["title"])
    for block in node.get("blocks", []):
        if block.get("text"):
            values.append(block["text"])
        for item in block.get("items", []):
            if item.get("text"):
                values.append(item["text"])
        for row in block.get("rows", []):
            values.extend(cell for cell in row if cell)
    for child in node.get("children", []):
        values.extend(collect_json_plain_text_for_bigrams(child))
    return values

## Metric 1: Text Bigram Similarity

Compares normalized bigram counts from the source DOCX text and the parsed JSON text. This helps evaluate whether the JSON dictionary preserves the source document's text content.

In [3]:
docx_bigrams = bigram_counts(collect_docx_plain_text(sample_path))
json_bigrams = bigram_counts(collect_json_plain_text_for_bigrams(parsed))

bigram_overlap = sum((docx_bigrams & json_bigrams).values())
docx_bigram_total = sum(docx_bigrams.values())
json_bigram_total = sum(json_bigrams.values())

bigram_recall = bigram_overlap / docx_bigram_total 
bigram_precision = bigram_overlap / json_bigram_total 
bigram_f1 = (
    2 * bigram_precision * bigram_recall / (bigram_precision + bigram_recall)
    if (bigram_precision + bigram_recall)
    else 0.0
)

missing_bigrams = list((docx_bigrams - json_bigrams).items())[:10]
extra_bigrams = list((json_bigrams - docx_bigrams).items())[:10]

print_score(
    "Text bigram similarity",
    bigram_f1,
    details=(
        f"Precision: {bigram_precision:.2%}\n"
        f"Recall: {bigram_recall:.2%}\n"
        f"DOCX bigrams: {docx_bigram_total}\n"
        f"JSON bigrams: {json_bigram_total}\n"
        f"Missing bigrams sample: {missing_bigrams}\n"
        f"Extra bigrams sample: {extra_bigrams}"
    ),
)


PASS: Text bigram similarity
Score: 100.00%
Precision: 100.00%
Recall: 100.00%
DOCX bigrams: 1548
JSON bigrams: 1548
Missing bigrams sample: []
Extra bigrams sample: []


## Metric 2: Schema Completeness

Checks whether the parsed dictionary includes the required top-level JSON keys.

In [4]:
expected_keys = {"title", "level", "path", "blocks", "children", "comments"}
actual_keys = set(parsed.keys())
matched = expected_keys & actual_keys
score = len(matched) / len(expected_keys)
missing = sorted(expected_keys - actual_keys)
print_score("Schema completeness", score, details=f"Missing keys: {missing}")

PASS: Schema completeness
Score: 100.00%
Missing keys: []


## Metric 3: Heading Tree Recall

Compares the expected heading paths against the heading paths found in the parsed JSON.

In [5]:
expected_heading_paths = {
    ("Red List Assessment",),
    ("Red List Assessment", "Assessment Information"),
    ("Red List Assessment", "Assessment Rationale"),
    ("Distribution",),
    ("Distribution", "Geographic Range"),
    ("Distribution", "Extent of Occurrence (EOO)"),
    ("Occurrence",),
    ("Population",),
    ("Habitats and Ecology",),
    ("Habitats and Ecology", "Systems"),
    ("Use and Trade",),
    ("Threats",),
    ("Conservation",),
    ("Conservation", "Research Needed"),
    ("Ecosystem Services",),
    ("Bibliography",),
}
actual_heading_paths = set(flatten_section_paths(parsed))
matched = expected_heading_paths & actual_heading_paths
score = len(matched) / len(expected_heading_paths)
missing = sorted(expected_heading_paths - actual_heading_paths)
print_score("Heading tree recall", score, details=f"Missing expected headings: {missing}")

PASS: Heading tree recall
Score: 100.00%
Missing expected headings: []


## Metric 4: Top-Level Heading Order Accuracy

Checks whether the top-level sections appear in the expected order.

In [6]:
expected_top_headings = [
    "Red List Assessment",
    "Distribution",
    "Occurrence",
    "Population",
    "Habitats and Ecology",
    "Use and Trade",
    "Threats",
    "Conservation",
    "Ecosystem Services",
    "Bibliography",
]
actual_top_headings = [child["title"] for child in parsed.get("children", [])]
matches = sum(a == e for a, e in zip(actual_top_headings, expected_top_headings))
score = matches / len(expected_top_headings)
print_score("Top-level heading order accuracy", score, details=f"Actual headings: {actual_top_headings}")

PASS: Top-level heading order accuracy
Score: 100.00%
Actual headings: ['Red List Assessment', 'Distribution', 'Occurrence', 'Population', 'Habitats and Ecology', 'Use and Trade', 'Threats', 'Conservation', 'Ecosystem Services', 'Bibliography']


## Metric 5: Block Type Distribution Accuracy

Checks whether the parsed JSON contains the expected number of paragraph, table, and style blocks for this sample.

In [7]:
expected_counts = Counter({"paragraph": 31, "table": 17, "style": 12})
actual_counts = Counter(block.get("type") for _, block in flatten_blocks(parsed))
total_expected = sum(expected_counts.values())
total_error = sum(abs(actual_counts[k] - expected_counts[k]) for k in expected_counts)
score = 1 - (total_error / total_expected)
print_score("Block type distribution accuracy", score, details=f"Expected: {dict(expected_counts)}\nActual: {dict(actual_counts)}")

PASS: Block type distribution accuracy
Score: 100.00%
Expected: {'paragraph': 31, 'table': 17, 'style': 12}
Actual: {'paragraph': 31, 'table': 17, 'style': 12}


## Metric 6: Plain Text Exact-Match Recall

Checks whether selected important plain-text values are exactly present in the parsed output.

In [8]:
expected_text_values = {
    "Draft",
    "Myrcia neosmithii - K.Campbell & K.Samra",
    "Common Names: No Common Names\nSynonyms: Calyptranthes smithii McVaugh",
    "Date of Assessment: 2021-06-29",
    "Assessor(s): Grice, H., Lucas, E. & Burton, G.",
    "Reviewer(s): Clarke, H.D.",
    "Regions: Global",
    "System: Terrestrial",
}
actual_text_values = set(collect_text_values(parsed))
matched = expected_text_values & actual_text_values
score = len(matched) / len(expected_text_values)
missing = sorted(expected_text_values - actual_text_values)
print_score("Plain text exact-match recall", score, details=f"Missing text values: {missing}")

PASS: Plain text exact-match recall
Score: 100.00%
Missing text values: []


## Metric 7: Rich-Text Formatting Recall

Checks whether selected expected rich-text strings are present in `text_rich` or `rows_rich` values.

In [9]:
expected_rich_values = {
    "<b>Draft</b>",
    "<b><i>Myrcia neosmithii</i></b><b> - K.Campbell & K.Samra</b>",
    "<b>Common Names: </b>No Common Names\n<b>Synonyms: </b>Calyptranthes smithii McVaugh",
    "<b>Date of Assessment: </b>2021-06-29",
    "<b>Reviewed?</b>",
    "<b>Status:</b>",
    "<b>System: </b>Terrestrial",
}
actual_rich_values = set(collect_rich_values(parsed))
matched = expected_rich_values & actual_rich_values
score = len(matched) / len(expected_rich_values)
missing = sorted(expected_rich_values - actual_rich_values)
print_score("Rich-text formatting recall", score, details=f"Missing rich-text values: {missing}")

PASS: Rich-text formatting recall
Score: 100.00%
Missing rich-text values: []


## Metric 8: Rich-Text Tag Coverage

Checks whether key rich-text tags are found anywhere in the parsed rich-text output.

In [10]:
all_rich_text = "\n".join(collect_rich_values(parsed))
expected_tags = {"<b>", "</b>", "<i>", "</i>", "<sup>", "</sup>"}
present_tags = {tag for tag in expected_tags if tag in all_rich_text}
score = len(present_tags) / len(expected_tags)
print_score("Rich-text tag coverage", score, details=f"Present tags: {sorted(present_tags)}")

PASS: Rich-text tag coverage
Score: 100.00%
Present tags: ['</b>', '</i>', '</sup>', '<b>', '<i>', '<sup>']


## Metric 9: Table Cell Exact-Match Accuracy

Checks selected table cells from known sections and reports exact-match accuracy across those cells.

In [11]:
table_expectations = [
    {
        "path": ["Red List Assessment", "Assessment Information"],
        "table_index": 0,
        "rows": [
            ["Reviewed?", "Date of Review:", "Status:", "Reasons for Rejection:", "Improvements Needed:"],
            ["true", "2021-08-17", "Passed", "-", "-"],
        ],
    },
    {
        "path": ["Distribution", "Extent of Occurrence (EOO)"],
        "table_index": 0,
        "rows": [
            ["Estimated extent of occurrence (EOO)- in km2", "EOO estimate calculated from Minimum Convex Polygon", "Justification"],
            ["8-6000", "-", "The EOO is highly uncertain. Lower bound based on minimal AOO for two collections. Upper bound based on approximate MCP for the Kanashen Amerindian Protected Area."],
        ],
    },
]

matched_cells = 0
total_cells = 0
missing_details = []

for expectation in table_expectations:
    section = find_section(parsed, expectation["path"])
    tables = [block for block in section.get("blocks", []) if block.get("type") == "table"] if section else []
    actual_rows = tables[expectation["table_index"]].get("rows", []) if len(tables) > expectation["table_index"] else []
    for row_index, expected_row in enumerate(expectation["rows"]):
        for col_index, expected_cell in enumerate(expected_row):
            total_cells += 1
            actual_cell = actual_rows[row_index][col_index] if row_index < len(actual_rows) and col_index < len(actual_rows[row_index]) else None
            if actual_cell == expected_cell:
                matched_cells += 1
            else:
                missing_details.append((expectation["path"], row_index, col_index, expected_cell, actual_cell))

score = matched_cells / total_cells
print_score("Table cell exact-match accuracy", score, details=f"Mismatches: {missing_details}")

PASS: Table cell exact-match accuracy
Score: 100.00%
Mismatches: []


## Metric 10: Style Feature Recall

Checks whether selected expected bold and italic snippets appear in final style blocks.

In [12]:
expected_style_features = {
    ("bold", "Draft"),
    ("bold", "Myrcia neosmithii"),
    ("bold", "Date of Assessment:"),
    ("bold", "System:"),
    ("italic", "Myrcia neosmithii"),
    ("italic", "Annals of the Missouri Botanical Garden"),
    ("italic", "Global Environmental Change"),
}
actual_style_features = collect_style_features(parsed)
matched = expected_style_features & actual_style_features
score = len(matched) / len(expected_style_features)
missing = sorted(expected_style_features - actual_style_features)
print_score("Style feature recall", score, details=f"Missing style features: {missing}")

PASS: Style feature recall
Score: 100.00%
Missing style features: []


## Metric 11: Comment Output Match

Checks that the sample document produces the expected comment count.

In [13]:
expected_comment_count = 0
actual_comment_count = len(parsed.get("comments", []))
score = 1.0 if actual_comment_count == expected_comment_count else 0.0
print_score("Comment output match", score, details=f"Expected comments: {expected_comment_count}\nActual comments: {actual_comment_count}")

PASS: Comment output match
Score: 100.00%
Expected comments: 0
Actual comments: 0


## Overall Parser Evaluation Score

Combines the main metric scores into a simple average. This gives a quick regression signal for this sample document.

In [14]:
metric_scores = {
    "text_bigram_similarity": bigram_f1,
    "schema_completeness": len(expected_keys & actual_keys) / len(expected_keys),
    "heading_tree_recall": len(expected_heading_paths & actual_heading_paths) / len(expected_heading_paths),
    "top_level_heading_order": sum(a == e for a, e in zip(actual_top_headings, expected_top_headings)) / len(expected_top_headings),
    "block_type_distribution": 1 - (total_error / total_expected),
    "plain_text_recall": len(expected_text_values & actual_text_values) / len(expected_text_values),
    "rich_text_recall": len(expected_rich_values & actual_rich_values) / len(expected_rich_values),
    "rich_text_tag_coverage": len(present_tags) / len(expected_tags),
    "table_cell_accuracy": matched_cells / total_cells,
    "style_feature_recall": len(expected_style_features & actual_style_features) / len(expected_style_features),
    "comment_output_match": score,
}

overall_score = sum(metric_scores.values()) / len(metric_scores)
for metric_name, metric_score in metric_scores.items():
    print(f"{metric_name}: {metric_score:.2%}")

print()
print_score("Overall parser evaluation score", overall_score)

text_bigram_similarity: 100.00%
schema_completeness: 100.00%
heading_tree_recall: 100.00%
top_level_heading_order: 100.00%
block_type_distribution: 100.00%
plain_text_recall: 100.00%
rich_text_recall: 100.00%
rich_text_tag_coverage: 100.00%
table_cell_accuracy: 100.00%
style_feature_recall: 100.00%
comment_output_match: 100.00%

PASS: Overall parser evaluation score
Score: 100.00%
